# ECE 285 Homework 1 — build, understand, and train a chest X-ray VAE

**This notebook is self-contained.** Download this `.ipynb` alone and open it
in local Jupyter/VS Code, Colab, Kaggle, or a remote GPU notebook. The code
cells contain the full Python implementation and write it into a separate
working directory. They do not need a GitHub login, a repository clone, or
the lecture video. Python 3.12 is a practical starting environment.

The homework asks for a VAE trained on
[Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia),
fresh generated images, and a **two-page NeurIPS-format report** with code.
We prepare hyperparameters, loss curves, Inception Score, FID, and images.

**How to use this notebook:**

1. Choose the machine/kernel below, then run the setup and teaching cells.
2. Read the tensor shapes, sampling, and loss explanations as you run them.
3. Set a dataset path or explicitly enable the Kaggle download.
4. Enable a two-epoch pilot, inspect it, then enable resume for the full run.
5. Enable generation/evaluation and export the results for your report.

Default **Run All** performs offline CPU/GPU checks and a tiny synthetic
learning demonstration. It does not download data, train on real images,
download Inception weights, or report fabricated experimental results.
Those stages have explicit switches in the configuration cell.


## 1. Choose where the Python kernel runs

A browser on your laptop can display a notebook whose Python process runs
on another machine. The kernel's machine determines which GPU you can use.
The workspace inspected for this implementation is **GitHub Codespaces on
Azure**, with a Microsoft virtual display adapter and no NVIDIA device.
Installing CUDA in that Codespace cannot attach your laptop's RTX 4080.

| Environment | Preparation | Kernel choice |
| --- | --- | --- |
| Laptop with RTX 4080 | Run Jupyter locally in Windows or WSL2/Linux; install NVIDIA driver and CUDA-enabled PyTorch | Your local `Python (ECE285 VAE)` kernel |
| Colab | Upload this notebook; choose a GPU runtime in Runtime → Change runtime type | Colab's GPU Python kernel |
| Kaggle | Upload/import this notebook; enable a GPU accelerator and Internet for downloads; optionally attach the assigned dataset | Kaggle's GPU Python kernel |
| Remote GPU/JupyterHub | Open/upload the notebook on the allocated GPU node | Kernel on that GPU node |
| Current Codespace / CPU | Use it to study and verify the implementation | `Python (ECE285 VAE)` here; CPU checks only by default |

On your laptop, create a fresh environment rather than copying `.venv`
from the Codespace. In a terminal in your local copy of the repository:

**Windows PowerShell:**

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
# Run the CUDA-enabled torch/torchvision command from the official selector
# below, using .\.venv\Scripts\python.exe -m pip instead of pip.
.\.venv\Scripts\python.exe -m pip install -r hw1_vae/requirements-notebook.txt
.\.venv\Scripts\python.exe -m ipykernel install --user --name ece285-vae --display-name "Python (ECE285 VAE)"
.\.venv\Scripts\python.exe -m jupyter lab hw1_vae/HW1_VAE.ipynb
```

**Linux / WSL2:**

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
# First run the CUDA-enabled torch/torchvision installation command below.
python -m pip install -r hw1_vae/requirements-notebook.txt
python -m ipykernel install --user --name ece285-vae --display-name "Python (ECE285 VAE)"
python -m jupyter lab hw1_vae/HW1_VAE.ipynb
```

Use the [official PyTorch installation selector](https://pytorch.org/get-started/locally/)
for a matching CUDA-enabled `torch` and `torchvision` pair. If your current
environment has CPU-only PyTorch, replace that pair first; an ordinary
install can leave an already satisfied CPU package in place. Restart the
notebook kernel after replacing PyTorch. A separate CUDA toolkit is not
required for ordinary prebuilt-wheel training; the GPU driver must work.
In WSL, follow [NVIDIA's WSL driver instructions](https://docs.nvidia.com/cuda/wsl-user-guide/index.html).

If you have only this notebook, install Jupyter with `python -m pip install
jupyterlab ipykernel`, open it, then use the dependency cell below. Colab
and Kaggle already provide the notebook interface and usually PyTorch.
Do not replace their working PyTorch installation unnecessarily.

References: [Jupyter installation](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html),
[kernel registration](https://ipython.readthedocs.io/en/stable/install/kernel_install.html),
[Codespaces GPU machine deprecation](https://github.blog/changelog/2025-08-01-upcoming-deprecation-of-gpu-machine-type-in-codespaces/).


## 2. Configure this experiment

Edit this cell before running later stages. Paths can point to a local disk,
an attached Kaggle dataset, or a mounted persistent directory. An empty
`DATA_DIR` means the teaching cells still run and data-dependent cells skip.

The isolated `WORKSPACE` holds generated Python source and results. When a
repository is detected it defaults to `runs/notebook_workspace`; elsewhere
it is `ece285_vae_notebook` under the starting directory. Source cells below
overwrite only their generated copies. To edit the model, change its source
cell, then restart and rerun from the top so imports do not use stale code.

Use one training switch at a time: `RUN_TRAINING` starts a **new** pilot;
`RUN_RESUME` extends an existing run to `TOTAL_EPOCHS` total epochs. After a
successful pilot, set `RUN_TRAINING=False` before enabling `RUN_RESUME`.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

INSTALL_MISSING_DEPENDENCIES = False  # Set True on a new runtime if needed.
WORKSPACE_OVERRIDE = None  # e.g. r"D:\ECE285_notebook" or "/content/ece285_vae_notebook"
DATA_DIR = None  # e.g. r"D:\datasets\chest_xray" or "/kaggle/input/chest-xray-pneumonia/chest_xray"
DOWNLOAD_DATA = False
RUN_TRAINING = False
RUN_RESUME = False
RUN_GENERATION = False
RUN_EVALUATION = False  # First use downloads ~104 MB of Inception weights.
EXPORT_RESULTS = False

DEVICE_REQUEST = "auto"  # "auto", "cuda", or "cpu"; auto selects CUDA if visible.
ALLOW_CPU_TRAINING = False  # Change deliberately for a CPU experiment.
IMAGE_SIZE = 64
LATENT_DIM = 128
BASE_CHANNELS = 32
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
KL_WEIGHT = 0.001
KL_WARMUP_EPOCHS = 0
SEED = 42
NUM_WORKERS = 0  # Portable baseline; try 2–4 on Linux after the pilot works.
CPU_THREADS = 2
PILOT_EPOCHS = 2
TOTAL_EPOCHS = 50
RUN_NAME = "baseline64"
RUN_DIR_OVERRIDE = None  # Prefer persistent storage for Colab/remote jobs.
NUM_GENERATED = 64
EVAL_NUM_SAMPLES = 0  # 0 = all official test images, matched generated count.

# Preserve the starting directory when this cell is rerun in one kernel.
if "_ECE285_INITIAL_CWD" not in globals():
    _ECE285_INITIAL_CWD = Path.cwd().resolve()
repository = next((p for p in (_ECE285_INITIAL_CWD, *_ECE285_INITIAL_CWD.parents)
                   if (p / "hw1_vae" / "model.py").is_file()
                   and (p / ".git").is_dir()), None)
default_workspace = (repository / "runs" / "notebook_workspace" if repository
                     else _ECE285_INITIAL_CWD / "ece285_vae_notebook")
WORKSPACE = Path(WORKSPACE_OVERRIDE).expanduser().resolve() if WORKSPACE_OVERRIDE else default_workspace
if repository and WORKSPACE.resolve() == repository.resolve():
    raise ValueError("Choose a separate notebook workspace, not the repository root.")
WORKSPACE.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "hw1_vae").mkdir(exist_ok=True)
os.chdir(WORKSPACE)
RUN_DIR = (Path(RUN_DIR_OVERRIDE).expanduser().resolve() if RUN_DIR_OVERRIDE
           else WORKSPACE / "runs" / RUN_NAME)
if DATA_DIR is not None:
    DATA_DIR = Path(DATA_DIR).expanduser()
    if not DATA_DIR.is_absolute():
        DATA_DIR = (_ECE285_INITIAL_CWD / DATA_DIR).resolve()
if RUN_TRAINING and RUN_RESUME:
    raise ValueError("Enable either a fresh pilot or resume, one at a time.")
print("Kernel Python:", sys.executable)
print("Generated source/workspace:", WORKSPACE)
print("Experiment outputs:", RUN_DIR)


The next cell installs only missing/too-old dependencies when the switch is
enabled, using **this kernel's** Python. It does not select a CUDA wheel or
install drivers. If packages were already imported when you upgraded them,
restart the kernel before continuing.


In [ ]:
DEPENDENCY_SPECS = ['torch>=2.6', 'torchvision>=0.21', 'numpy>=1.26', 'Pillow>=10', 'matplotlib>=3.8', 'pytorch-ignite>=0.5.5,<0.6', 'scipy>=1.12', 'kagglehub>=0.3']

import importlib.metadata
from packaging.requirements import Requirement

missing = []
for spec in DEPENDENCY_SPECS:
    requirement = Requirement(spec)
    try:
        installed = importlib.metadata.version(requirement.name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed is None or not requirement.specifier.contains(installed, prereleases=True):
        missing.append(spec)
if missing and INSTALL_MISSING_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
elif missing:
    raise RuntimeError("Missing/old packages: " + ", ".join(missing)
                       + ". Install the correct PyTorch build first if needed, then set "
                       "INSTALL_MISSING_DEPENDENCIES=True and rerun this cell.")
else:
    print("All VAE dependencies meet the required versions.")

def run_module(module, *arguments):
    # Run the displayed package with this kernel's Python.
    command = [sys.executable, "-u", "-m", module, *map(str, arguments)]
    print("Running:", subprocess.list2cmdline(command), flush=True)
    environment = dict(os.environ)
    environment["OMP_NUM_THREADS"] = str(CPU_THREADS)
    environment["MKL_NUM_THREADS"] = str(CPU_THREADS)
    environment["PYTHONPATH"] = str(WORKSPACE)
    # Stream output in Jupyter rather than buffering a long training run.
    process = subprocess.Popen(command, cwd=WORKSPACE, env=environment,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


## 3. Check the actual compute device

These source cells create the small package used by the rest of the
notebook. `%%writefile` writes the visible code to the isolated workspace;
it does not train or change your repository implementation.

The diagnostic separates four questions: is NVIDIA hardware exposed, does
a driver respond, was PyTorch built with CUDA, and can a real CUDA operation
plus VAE backpropagation execute? Seeing an RTX 4080 in your laptop's Task
Manager does not imply a remote kernel can use it.


In [ ]:
%%writefile hw1_vae/__init__.py
"""Homework 1: an unconditional convolutional chest X-ray VAE."""


In [ ]:
%%writefile hw1_vae/model.py
"""The encoder, reparameterization trick, decoder, and lecture-style VAE loss."""

import torch
from torch import Tensor, nn
from torch.nn import functional as F


class ConvVAE(nn.Module):
    """Map a grayscale image to a diagonal Gaussian, then back to an image.

    Four stride-two convolutions halve the spatial dimensions four times.
    For 64x64 input: 1x64x64 -> 32x32x32 -> 64x16x16 -> 128x8x8
    -> 256x4x4. The decoder mirrors all four stages, including the last one.
    Normalization layers are omitted to keep this baseline simple and to
    allow a final batch containing just one image.
    """

    def __init__(self, image_size: int = 64, latent_dim: int = 128,
                 base_channels: int = 32):
        super().__init__()
        if image_size < 16 or image_size % 16:
            raise ValueError("image_size must be a positive multiple of 16 (default: 64).")
        if latent_dim < 1 or base_channels < 1:
            raise ValueError("latent_dim and base_channels must be positive.")
        self.image_size = image_size
        self.latent_dim = latent_dim
        self.base_channels = base_channels
        self.spatial_size = image_size // 16
        channels = [base_channels * m for m in (1, 2, 4, 8)]

        encoder = []
        in_channels = 1
        for out_channels in channels:
            encoder.extend([
                nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
                nn.LeakyReLU(0.2, inplace=True),
            ])
            in_channels = out_channels
        self.encoder = nn.Sequential(*encoder)
        flat_size = channels[-1] * self.spatial_size ** 2
        # These are different learned heads, not mean/variance of the image batch.
        self.fc_mu = nn.Linear(flat_size, latent_dim)
        self.fc_logvar = nn.Linear(flat_size, latent_dim)
        self.decoder_input = nn.Linear(latent_dim, flat_size)

        decoder = []
        for in_channels, out_channels in zip(channels[::-1], channels[-2::-1]):
            decoder.extend([
                nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1),
                nn.LeakyReLU(0.2, inplace=True),
            ])
        decoder.extend([nn.ConvTranspose2d(channels[0], 1, 4, 2, 1), nn.Sigmoid()])
        self.decoder = nn.Sequential(*decoder)

    def encode(self, x: Tensor) -> tuple[Tensor, Tensor]:
        expected = (1, self.image_size, self.image_size)
        if x.ndim != 4 or tuple(x.shape[1:]) != expected:
            raise ValueError(f"Expected [batch, {expected}], received {tuple(x.shape)}.")
        features = self.encoder(x).flatten(start_dim=1)
        # log(variance) may take any real value; exp(logvar) is always positive.
        return self.fc_mu(features), self.fc_logvar(features)

    @staticmethod
    def reparameterize(mu: Tensor, logvar: Tensor) -> Tensor:
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        return mu + std * epsilon

    def decode(self, z: Tensor) -> Tensor:
        features = self.decoder_input(z)
        features = features.reshape(-1, self.base_channels * 8,
                                    self.spatial_size, self.spatial_size)
        return self.decoder(features)

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(reconstruction: Tensor, target: Tensor, mu: Tensor,
             logvar: Tensor, kl_weight: float = 0.001) -> dict[str, Tensor]:
    """MSE averaged over pixels + weighted KL summed over latent coordinates.

    Matches the practical loss displayed at about 66:00 in the supplied lecture.
    KL is measured in nats per image. This weighted training objective must not
    be labeled an unscaled, unit-weight negative ELBO. Changing MSE from mean
    to sum changes the relative KL strength by the number of image pixels.
    """
    if reconstruction.shape != target.shape:
        raise ValueError("Reconstruction and input must have identical shapes.")
    if kl_weight < 0:
        raise ValueError("kl_weight cannot be negative.")
    reconstruction_loss = F.mse_loss(reconstruction, target, reduction="mean")
    kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1).mean()
    return {"total": reconstruction_loss + kl_weight * kl,
            "reconstruction": reconstruction_loss, "kl": kl}


In [ ]:
%%writefile hw1_vae/check_compute.py
"""Inspect the current Python host and verify CUDA without changing its setup.

Run ``python -m hw1_vae.check_compute --test-vae --require-cuda`` on the
machine that should train. The check distinguishes missing host GPU access
from a CPU-only PyTorch installation. It never installs drivers or packages.
"""

from __future__ import annotations

import argparse
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path


def _pci_devices() -> list[dict[str, str]]:
    """Linux-only display/NVIDIA PCI metadata; empty on other platforms."""
    devices = []
    for device in sorted(Path("/sys/bus/pci/devices").glob("*")):
        try:
            vendor = (device / "vendor").read_text().strip()
            device_class = (device / "class").read_text().strip()
            if vendor == "0x10de" or device_class.startswith("0x03"):
                devices.append({"pci_address": device.name, "vendor": vendor,
                                "device": (device / "device").read_text().strip(),
                                "class": device_class})
        except OSError:
            continue
    return devices


def collect_report(test_vae: bool = False) -> dict:
    """Return JSON-safe evidence and, where possible, execute GPU operators."""
    executable = shutil.which("nvidia-smi")
    wsl_executable = Path("/usr/lib/wsl/lib/nvidia-smi")
    if not executable and wsl_executable.is_file():
        executable = str(wsl_executable)
    smi = {"found": bool(executable), "ok": False, "gpus": []}
    if executable:
        try:
            result = subprocess.run(
                [executable, "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=15, check=False,
            )
            smi["ok"] = result.returncode == 0
            if smi["ok"]:
                smi["gpus"] = [line.strip() for line in result.stdout.splitlines() if line.strip()]
            else:
                smi["error"] = (result.stderr or result.stdout).strip()[:1000]
        except (OSError, subprocess.TimeoutExpired) as error:
            smi["error"] = str(error)
    pci = _pci_devices()
    report = {
        "python_executable": sys.executable,
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "codespaces": os.environ.get("CODESPACES", "").lower() == "true",
        "wsl": "microsoft" in platform.release().lower() or bool(os.environ.get("WSL_DISTRO_NAME")),
        "nvidia_device_files": sorted(str(path) for path in Path("/dev").glob("nvidia*")),
        "linux_nvidia_driver_file": Path("/proc/driver/nvidia/version").is_file(),
        "display_and_nvidia_pci_devices": pci,
        "nvidia_smi": smi,
        "torch_installed": False,
        "cuda_available": False,
        "cuda_tensor_test": {"status": "skipped"},
        "vae_test": {"status": "skipped", "requested": test_vae},
        "cuda_ready": False,
    }
    try:
        import torch
        report["torch_installed"] = True
        report["torch_version"] = str(torch.__version__)
        report["torch_cuda_build"] = torch.version.cuda
        report["cuda_available"] = torch.cuda.is_available()
        report["cuda_device_count"] = torch.cuda.device_count()
        report["cuda_devices"] = []
        if report["cuda_available"]:
            for index in range(torch.cuda.device_count()):
                properties = torch.cuda.get_device_properties(index)
                report["cuda_devices"].append({
                    "index": index, "name": properties.name,
                    "total_memory_gib": round(properties.total_memory / 1024**3, 2),
                    "compute_capability": list(torch.cuda.get_device_capability(index)),
                })
            try:
                tensor = torch.randn(16, 16, device="cuda", requires_grad=True)
                loss = (tensor @ tensor.T).square().mean()
                loss.backward()
                torch.cuda.synchronize()
                if not torch.isfinite(loss).item() or not torch.isfinite(tensor.grad).all().item():
                    raise FloatingPointError("CUDA tensor check produced nonfinite values")
                report["cuda_tensor_test"] = {"status": "passed", "device": str(tensor.device)}
                del tensor, loss
            except Exception as error:
                report["cuda_tensor_test"] = {"status": "failed", "error": str(error)}
            if test_vae and report["cuda_tensor_test"]["status"] == "passed":
                try:
                    from .model import ConvVAE, vae_loss
                    model = ConvVAE(image_size=64, latent_dim=8, base_channels=8).to("cuda")
                    images = torch.rand(2, 1, 64, 64, device="cuda")
                    reconstruction, mu, logvar = model(images)
                    loss = vae_loss(reconstruction, images, mu, logvar)["total"]
                    loss.backward()
                    torch.cuda.synchronize()
                    if reconstruction.shape != images.shape or not torch.isfinite(loss).item():
                        raise FloatingPointError("VAE CUDA check returned invalid output or loss")
                    if not all(parameter.grad is not None and torch.isfinite(parameter.grad).all().item()
                               for parameter in model.parameters()):
                        raise FloatingPointError("VAE CUDA gradients are missing or nonfinite")
                    report["vae_test"] = {"status": "passed", "requested": True,
                                          "device": str(reconstruction.device),
                                          "reconstruction_shape": list(reconstruction.shape)}
                    del model, images, reconstruction, mu, logvar, loss
                except Exception as error:
                    report["vae_test"] = {"status": "failed", "requested": True, "error": str(error)}
            report["cuda_ready"] = (
                report["cuda_tensor_test"]["status"] == "passed"
                and (not test_vae or report["vae_test"]["status"] == "passed")
            )
    except ImportError as error:
        report["torch_import_error"] = str(error)
    except Exception as error:
        report["torch_probe_error"] = str(error)

    hardware_signal = (bool(report["nvidia_device_files"])
                       or report["linux_nvidia_driver_file"] or smi["ok"]
                       or any(device["vendor"] == "0x10de" for device in pci))
    report["nvidia_host_signals_present"] = hardware_signal
    if report["cuda_ready"]:
        report["conclusion"] = "CUDA is usable by this Python process; the requested execution checks passed."
    elif report["codespaces"] and not hardware_signal and not report["cuda_available"]:
        report["conclusion"] = (
            "This process is running in a cloud Codespace with no exposed NVIDIA GPU/driver signals. "
            "The laptop GPU is on a different host. Run the notebook using a Python kernel on the "
            "laptop or a GPU compute server; installing CUDA packages here cannot attach that hardware."
        )
    elif report.get("torch_cuda_build") is None and report["torch_installed"]:
        report["conclusion"] = (
            "PyTorch is a CPU-only build. On a host exposing an NVIDIA GPU and compatible driver, "
            "install a matching CUDA-enabled torch/torchvision pair, restart the kernel, and check again."
        )
    elif not report["torch_installed"]:
        report["conclusion"] = "PyTorch could not be imported. Install PyTorch in the kernel's Python environment."
    else:
        report["conclusion"] = (
            "CUDA is not ready. Inspect the driver/host visibility, PyTorch build and execution-test "
            "results above; see GPU_SETUP.md for the setup paths."
        )
    return report


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output", type=Path, help="Also save the evidence as a JSON file")
    parser.add_argument("--require-cuda", action="store_true", help="Exit nonzero unless CUDA execution succeeds")
    parser.add_argument("--test-vae", action="store_true", help="Also verify a small VAE forward/backward on CUDA")
    args = parser.parse_args()
    report = collect_report(test_vae=args.test_vae)
    serialized = json.dumps(report, indent=2, allow_nan=False) + "\n"
    print(serialized, end="")
    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(serialized)
    failed = any(report[key]["status"] == "failed" for key in ("cuda_tensor_test", "vae_test"))
    if failed:
        raise SystemExit(1)
    if args.require_cuda and not report["cuda_ready"]:
        raise SystemExit(2)


if __name__ == "__main__":
    main()


In [ ]:
# Prefer the generated source even if another hw1_vae was imported earlier.
for name in tuple(sys.modules):
    if name == "hw1_vae" or name.startswith("hw1_vae."):
        del sys.modules[name]
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
import importlib
importlib.invalidate_caches()
from hw1_vae.check_compute import collect_report
from IPython.display import display, JSON, Image as DisplayImage, Markdown

compute_report = collect_report(test_vae=True)
display(JSON(compute_report))
(WORKSPACE / "compute_report.json").write_text(json.dumps(compute_report, indent=2) + "\n")
print(compute_report["conclusion"])


## 4. Understand the encoder, Gaussian sample, and decoder

The complete `ConvVAE` is visible in the `model.py` cell above. Its input
is a batch $x$ of grayscale images with shape $[B,1,64,64]$.

| Encoder operation | Default output shape |
| --- | --- |
| Conv2d(1,32,4,2,1), LeakyReLU | $[B,32,32,32]$ |
| Conv2d(32,64,4,2,1), LeakyReLU | $[B,64,16,16]$ |
| Conv2d(64,128,4,2,1), LeakyReLU | $[B,128,8,8]$ |
| Conv2d(128,256,4,2,1), LeakyReLU | $[B,256,4,4]$ |
| Flatten, then two independent linear heads | $\mu,\log\sigma^2 \in \mathbb R^{B\times128}$ |

Stride two halves the spatial dimensions. More channels summarize more
features. The two heads describe a **distribution for each image**:
$q_\phi(z|x)=\mathcal N(\mu_\phi(x),\operatorname{diag}(\sigma_\phi^2(x)))$.
They are not image-batch statistics or diagnostic labels.

To sample and retain gradients, use reparameterization:
$$\epsilon\sim\mathcal N(0,I),\qquad
  z=\mu+\exp(\tfrac12\log\sigma^2)\odot\epsilon.$$
The factor one-half converts log-variance into log-standard-deviation.
Backpropagation can then reach both Gaussian heads through $z$.

The decoder linearly expands $z$ into $256\times4\times4$, then applies
four transposed convolutions to double spatial size at each stage. Its
sigmoid produces $[B,1,64,64]$ values in $[0,1]$. We display its predicted
image mean rather than adding observation noise.

This is a smaller grayscale adaptation of the lecture example. The lecture
around 65–66 minutes uses RGB, five encoder stages, and latent size 1024.
Our four-stage mirror also avoids the apparent missing upsampling stage
on that displayed decoder slide.


In [ ]:
%%writefile hw1_vae/data.py
"""Load the homework's chest X-ray images without using the diagnostic labels.

The VAE learns from images alone. NORMAL and PNEUMONIA are both included;
their directory names are useful for checking that the dataset is complete.
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset


SPLITS = ("train", "val", "test")
CLASSES = ("NORMAL", "PNEUMONIA")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
KAGGLE_DATASET = "paultimothymooney/chest-xray-pneumonia"


def resolve_data_root(data_dir: str | Path) -> Path:
    """Find one root containing train/val/test, including common Kaggle wrappers.

    Some extracts contain a second copy under chest_xray/chest_xray. Selecting
    the first complete root, then reading only its exact split folders, avoids
    accidentally counting both copies or mixing training and test images.
    """
    supplied = Path(data_dir).expanduser().resolve()
    candidates = (supplied, supplied / "chest_xray", supplied / "chest_xray" / "chest_xray")
    for candidate in candidates:
        if all((candidate / split).is_dir() for split in SPLITS):
            return candidate
    checked = "\n".join(f"  {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find a chest X-ray dataset root containing train/, val/, and test/.\n"
        f"Checked:\n{checked}\n"
        "Pass --data-dir pointing at the extracted chest_xray directory, or run\n"
        "python -m hw1_vae.data --download to explicitly download the Kaggle dataset."
    )


def _is_image(path: Path, class_dir: Path) -> bool:
    relative_parts = path.relative_to(class_dir).parts
    return (
        path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
        and not any(part.startswith(".") or part == "__MACOSX" for part in relative_parts)
    )


class ChestXrayDataset(Dataset):
    """One official split, resized to grayscale tensors of shape [1, H, W].

    Preprocessing is deterministic for every split: convert to grayscale,
    bilinear-resize to a square, and scale uint8 intensities to [0, 1]. No
    augmentation or label conditioning changes the reconstruction target.
    """

    def __init__(self, data_dir: str | Path, split: str, image_size: int = 64) -> None:
        if split not in SPLITS:
            raise ValueError(f"split must be one of {SPLITS}; received {split!r}")
        if not isinstance(image_size, int) or image_size <= 0:
            raise ValueError("image_size must be a positive integer")
        self.root = resolve_data_root(data_dir)
        self.split = split
        self.image_size = image_size
        self.class_counts: dict[str, int] = {}
        image_paths: list[Path] = []
        for class_name in CLASSES:
            class_dir = self.root / split / class_name
            if not class_dir.is_dir():
                raise FileNotFoundError(
                    f"Missing dataset class directory: {class_dir}. "
                    "Each official split must contain NORMAL/ and PNEUMONIA/."
                )
            paths = sorted(path for path in class_dir.rglob("*") if _is_image(path, class_dir))
            if not paths:
                raise FileNotFoundError(f"No supported image files found in {class_dir}")
            self.class_counts[class_name] = len(paths)
            image_paths.extend(paths)
        self.image_paths = tuple(image_paths)

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int) -> torch.Tensor:
        path = self.image_paths[index]
        try:
            with Image.open(path) as image:
                image = image.convert("L").resize(
                    (self.image_size, self.image_size), resample=Image.Resampling.BILINEAR
                )
                pixels = np.array(image, dtype=np.float32, copy=True) / 255.0
        except (OSError, ValueError) as error:
            raise RuntimeError(f"Could not read chest X-ray image {path}: {error}") from error
        return torch.from_numpy(pixels).unsqueeze(0)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    source = parser.add_mutually_exclusive_group(required=True)
    source.add_argument("--data-dir", type=Path, help="Existing dataset root or its extraction parent")
    source.add_argument(
        "--download", action="store_true", help="Explicitly download the homework dataset with kagglehub"
    )
    args = parser.parse_args()
    if args.download:
        try:
            import kagglehub
        except ImportError:
            parser.error("--download requires kagglehub: python -m pip install kagglehub")
        data_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    else:
        data_dir = args.data_dir
    root = resolve_data_root(data_dir)
    print(f"Resolved dataset root: {root}")
    print(f"{'split':<8} {'NORMAL':>9} {'PNEUMONIA':>12} {'total':>9}")
    for split in SPLITS:
        dataset = ChestXrayDataset(root, split)
        print(
            f"{split:<8} {dataset.class_counts['NORMAL']:>9} "
            f"{dataset.class_counts['PNEUMONIA']:>12} {len(dataset):>9}"
        )
    print("Preprocessing: grayscale, bilinear resize to 64x64, float32 in [0, 1].")
    print("Use this resolved root as --data-dir for training and evaluation.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hw1_vae/utils.py
"""Small shared helpers for devices, checkpoints, and reproducible figures."""

import json
import math
import platform
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from .model import ConvVAE


def resolve_device(name: str = "auto") -> torch.device:
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Install a CUDA-enabled PyTorch build on the "
                           "GPU host, or explicitly use --device cpu for a small check.")
    return torch.device(name)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Reproducible kernels where available. Exact results can still vary by
    # PyTorch release/hardware; the run records versions and device information.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def save_json(path, data) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")


def environment_info(device: torch.device) -> dict:
    return {"python": platform.python_version(), "torch": str(torch.__version__),
            "cuda_build": torch.version.cuda, "device": str(device),
            "device_name": torch.cuda.get_device_name(device) if device.type == "cuda"
            else platform.processor() or "CPU"}


def load_model(checkpoint, device) -> tuple[ConvVAE, dict]:
    # This project's checkpoints contain tensors and primitive dictionaries.
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    config = state["config"]
    model = ConvVAE(**{key: config[key] for key in
                       ("image_size", "latent_dim", "base_channels")})
    model.load_state_dict(state["model_state"])
    return model.to(device).eval(), state


def save_grid(images: torch.Tensor, path, nrow: int = 8) -> None:
    """Save actual [0,1] intensities, without per-image contrast normalization."""
    images = images.detach().cpu().clamp(0, 1)
    if images.ndim != 4 or images.shape[1] != 1 or len(images) == 0:
        raise ValueError("save_grid expects a nonempty batch of grayscale images.")
    nrow = min(nrow, len(images))
    if nrow < 1:
        raise ValueError("nrow must be positive.")
    height, width = images.shape[-2:]
    padding = 2
    rows = math.ceil(len(images) / nrow)
    canvas = Image.new("L", (nrow * (width + padding) + padding,
                             rows * (height + padding) + padding), color=0)
    arrays = (images[:, 0].numpy() * 255).round().astype(np.uint8)
    for index, array in enumerate(arrays):
        canvas.paste(Image.fromarray(array),
                     (padding + (index % nrow) * (width + padding),
                      padding + (index // nrow) * (height + padding)))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path)


In [ ]:
import torch
import numpy as np
from PIL import Image
from hw1_vae.model import ConvVAE, vae_loss
from hw1_vae.data import ChestXrayDataset, resolve_data_root
from hw1_vae.utils import resolve_device, seed_everything, save_grid, load_model

torch.set_num_threads(CPU_THREADS)
seed_everything(SEED)
DEVICE = resolve_device(DEVICE_REQUEST)
print("Selected device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU teaching checks are enabled. Real CPU training requires ALLOW_CPU_TRAINING=True.")

demo_model = ConvVAE(IMAGE_SIZE, LATENT_DIM, BASE_CHANNELS).to(DEVICE)
demo_x = torch.rand(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
demo_reconstruction, demo_mu, demo_logvar = demo_model(demo_x)
print("Input:", tuple(demo_x.shape))
print("Encoded Gaussian heads:", tuple(demo_mu.shape), tuple(demo_logvar.shape))
print("Reconstruction:", tuple(demo_reconstruction.shape))
assert demo_reconstruction.shape == demo_x.shape
print("Parameters:", f"{sum(p.numel() for p in demo_model.parameters()):,}")


## 5. Derive and execute the training objective

Training balances image reconstruction against latent regularization:

$$R=\frac{1}{BD}\sum_{b=1}^B\|\hat x_b-x_b\|_2^2,\qquad
K=-\frac{1}{2B}\sum_{b,j}
(1+\log\sigma_{bj}^2-\mu_{bj}^2-\exp(\log\sigma_{bj}^2)),\qquad
L=R+\lambda K.$$

Here $D=64\times64$ pixels by default. The reconstruction term averages
over batch and pixels; KL sums over latent coordinates and averages over
images. $K=0$ for $\mu=0,\log\sigma^2=0$, the standard-normal prior.

MSE is motivated by a fixed-variance Gaussian observation model. The
practical lecture uses mean MSE plus `0.001 * KL`, which we retain as the
starting convention. State that exact weighted objective in your report;
its numeric value is not a unit-weight negative ELBO. Replacing mean MSE
by pixel-summed MSE multiplies that term by 4096 at this resolution and
changes the balance unless the KL weight is adjusted.

The following cell performs **one update on random synthetic pixels** to
show the exact mechanics. It discards this model afterward. The real
training command creates its own freshly initialized model or resumes a
checkpoint; it never uses this demonstration as a trained result.


In [ ]:
demo_optimizer = torch.optim.Adam(demo_model.parameters(), lr=LEARNING_RATE)
demo_optimizer.zero_grad(set_to_none=True)
losses = vae_loss(demo_reconstruction, demo_x, demo_mu, demo_logvar, KL_WEIGHT)
losses["total"].backward()
for name, head in (("mu", demo_model.fc_mu), ("logvar", demo_model.fc_logvar)):
    print(name, "head gradient norm:", head.weight.grad.norm().item())
    assert torch.isfinite(head.weight.grad).all()
demo_optimizer.step()
print("SYNTHETIC demonstration losses:", {k: v.item() for k, v in losses.items()})

zeros = torch.zeros(2, LATENT_DIM, device=DEVICE)
prior_check = vae_loss(demo_x, demo_x, zeros, zeros)
assert abs(prior_check["kl"].item()) < 1e-7
print("Analytic KL at the prior:", prior_check["kl"].item())
del demo_optimizer, demo_model, demo_x, demo_reconstruction, demo_mu, demo_logvar, losses
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 6. Prepare and inspect the assigned data

The loader includes **both** NORMAL and PNEUMONIA images. This is an
unconditional generator; the diagnostic labels are not training targets.
Each image becomes a grayscale tensor, bilinear-resized to the configured
square size, with uint8 intensity scaled to `[0,1]`.

Keep the official folders separate: `train` updates weights, `val` chooses
checkpoints, and `test` is reserved for final evaluation. The official
validation folder is tiny, so its loss is noisy. Preserving these folders
does not by itself establish patient-level independence or clinical validity.

Set `DATA_DIR` to the directory containing `train/NORMAL`,
`train/PNEUMONIA`, and matching `val`/`test` folders. Common extra
`chest_xray` wrappers and hidden Apple metadata files are handled. Or set
`DOWNLOAD_DATA=True` to call Kagglehub for the exact assigned dataset.
That switch uses network and disk and may require Kaggle authentication.
Do not paste credentials into a notebook that you will share.

On Kaggle, attach the dataset and point `DATA_DIR` at its mounted folder.
On Colab, download into the runtime or mount your own storage first. Runtime
disks can disappear, so keep run/checkpoint copies in persistent storage.


In [ ]:
if DOWNLOAD_DATA:
    import kagglehub
    DATA_DIR = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))

datasets = {}
if DATA_DIR is None:
    print("Dataset stage skipped. Set DATA_DIR or DOWNLOAD_DATA=True when ready.")
else:
    DATA_DIR = resolve_data_root(DATA_DIR)
    datasets = {split: ChestXrayDataset(DATA_DIR, split, IMAGE_SIZE)
                for split in ("train", "val", "test")}
    print("Resolved data root:", DATA_DIR)
    for split, dataset in datasets.items():
        print(split, "total:", len(dataset), "classes:", dataset.class_counts)
    first_images = torch.stack([datasets["train"][i]
                                for i in range(min(8, len(datasets["train"])))])
    input_grid = WORKSPACE / "input_preview.png"
    save_grid(first_images, input_grid, nrow=8)
    display(DisplayImage(filename=str(input_grid)))
    print("Real training inputs after preprocessing; not generated images.")


## 7. Inspect the complete training loop and run offline checks

The visible `train.py` cell below contains the full loop. Each training
batch zeroes gradients, samples a latent, reconstructs its inputs, computes
the loss, backpropagates, and steps Adam. Validation uses no gradient updates
and repeats the same latent noise each epoch to reduce sampling variation.

Losses are averaged by the number of images, including a short last batch.
`best.pt` is selected with validation objective at the target KL weight;
`last.pt` also carries optimizer state and RNG state for resuming. Optional
warmup changes the training KL weight while selection uses the final weight.
Source cells do not execute their CLI `main()`; only later command cells do.


In [ ]:
%%writefile hw1_vae/train.py
"""Train on Kaggle's train split; select checkpoints using its val split."""

import argparse
import csv
import hashlib
import math
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from .data import ChestXrayDataset
from .model import ConvVAE, vae_loss
from .utils import environment_info, resolve_device, save_grid, save_json, seed_everything


def run_epoch(model, loader, device, kl_weight, optimizer=None):
    """Average per-image losses, weighting a short final batch correctly."""
    training = optimizer is not None
    model.train(training)
    sums = {"total": 0.0, "reconstruction": 0.0, "kl": 0.0}
    count = 0
    with torch.set_grad_enabled(training):
        for images in loader:
            images = images.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            reconstruction, mu, logvar = model(images)
            losses = vae_loss(reconstruction, images, mu, logvar, kl_weight)
            if not all(torch.isfinite(value).item() for value in losses.values()):
                raise FloatingPointError("Nonfinite loss. Check input scaling and learning rate.")
            if training:
                losses["total"].backward()
                optimizer.step()
            for name, value in losses.items():
                sums[name] += value.item() * len(images)
            count += len(images)
    if not count:
        raise ValueError("The data loader is empty.")
    return {name: total / count for name, total in sums.items()}


def plot_history(history, path):
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
    for axis, key, label in zip(axes, ("total", "reconstruction", "kl"),
                                ("Weighted objective", "Pixel MSE", "KL (nats/image)")):
        for split in ("train", "val"):
            axis.plot([row["epoch"] for row in history],
                      [row[f"{split}_{key}"] for row in history], label=split)
        axis.set(xlabel="Epoch", ylabel=label)
        axis.grid(alpha=0.25)
        axis.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)


def data_manifest(datasets):
    """Track which relative filenames and file sizes belong to each split."""
    manifest = {}
    for split, dataset in datasets.items():
        manifest[split] = [
            {"file": path.relative_to(dataset.root).as_posix(), "bytes": path.stat().st_size}
            for path in dataset.image_paths
        ]
    # Not an image-content hash: it catches accidental changes of split/layout,
    # while allowing a run to resume after moving the data to another computer.
    import json
    digest = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest, digest


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, default=Path("runs/hw1_vae"))
    parser.add_argument("--epochs", type=int, default=50, help="Total target epochs, including resumed epochs.")
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--image-size", type=int, default=64)
    parser.add_argument("--latent-dim", type=int, default=128)
    parser.add_argument("--base-channels", type=int, default=32)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument("--kl-weight", type=float, default=0.001)
    parser.add_argument("--kl-warmup-epochs", type=int, default=0,
                        help="Optional linear ramp of training KL weight; validation always uses target weight.")
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--device", choices=("auto", "cpu", "cuda", "mps"), default="auto")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--resume", type=Path, help="Resume last.pt at the next epoch; saved hyperparameters win.")
    return parser.parse_args()


def main():
    args = parse_args()
    if args.epochs < 1 or args.num_workers < 0:
        raise ValueError("epochs must be positive and num-workers nonnegative.")
    device = resolve_device(args.device)
    resume = torch.load(args.resume, map_location="cpu", weights_only=True) if args.resume else None
    config = {key: str(value) if isinstance(value, Path) else value
              for key, value in vars(args).items()}
    if resume:
        for key in ("batch_size", "image_size", "latent_dim", "base_channels",
                    "learning_rate", "kl_weight", "kl_warmup_epochs", "seed"):
            config[key] = resume["config"][key]
    if (config["batch_size"] < 1 or config["learning_rate"] <= 0
            or config["kl_weight"] < 0 or config["kl_warmup_epochs"] < 0):
        raise ValueError("Invalid batch size, learning rate, or KL settings.")
    seed_everything(config["seed"])
    datasets = {split: ChestXrayDataset(args.data_dir, split, config["image_size"])
                for split in ("train", "val")}
    manifest, manifest_digest = data_manifest(datasets)
    if resume and resume["manifest_digest"] != manifest_digest:
        raise ValueError("Dataset filenames/sizes differ from the saved run; start a separate output directory.")
    output = args.output_dir
    occupied = any((output / name).exists() for name in ("last.pt", "best.pt", "history.csv"))
    if not resume and occupied:
        raise FileExistsError(f"{output} already contains a run. Use --resume or a new --output-dir.")
    if resume and occupied and output.resolve() != args.resume.parent.resolve():
        raise FileExistsError(f"{output} already contains a different run. Resume in the checkpoint's "
                              "directory or choose an empty --output-dir.")
    if resume and args.epochs <= resume["epoch"]:
        raise ValueError(f"Checkpoint is already at epoch {resume['epoch']}; --epochs must be larger.")
    previous_best = None
    if resume:
        candidate = args.resume.parent / "best.pt"
        if candidate.is_file():
            best_state = torch.load(candidate, map_location="cpu", weights_only=True)
            if (best_state["manifest_digest"] != resume["manifest_digest"]
                    or best_state["best_val"] != resume["best_val"]
                    or best_state["epoch"] > resume["epoch"]):
                raise ValueError("The sibling best.pt does not match the resumed checkpoint's "
                                 "selection history. Use the latest last.pt with its matching best.pt.")
            previous_best = candidate
        elif resume["history"][-1]["val_total"] == resume["best_val"]:
            previous_best = args.resume
        else:
            raise FileNotFoundError("The historical best checkpoint is missing. Keep best.pt next "
                                    "to last.pt when moving a run to another computer.")
    output.mkdir(parents=True, exist_ok=True)
    save_json(output / "config.json", config)
    save_json(output / "environment.json", environment_info(device))
    save_json(output / "data_manifest.json", manifest)
    print(f"Device: {device}; split sizes: "
          f"train={len(datasets['train'])}, val={len(datasets['val'])}", flush=True)
    if len(datasets["val"]) < 100:
        print("Validation split is small: checkpoint selection can be noisy. Keep test data held out.", flush=True)

    train_generator = torch.Generator().manual_seed(config["seed"])
    loaders = {
        split: DataLoader(dataset, batch_size=config["batch_size"], shuffle=split == "train",
                          num_workers=args.num_workers, pin_memory=device.type == "cuda",
                          generator=train_generator if split == "train" else None, drop_last=False)
        for split, dataset in datasets.items()
    }
    model = ConvVAE(**{key: config[key] for key in ("image_size", "latent_dim", "base_channels")}).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}", flush=True)
    history, best_val, start_epoch = [], math.inf, 1
    if resume:
        model.load_state_dict(resume["model_state"])
        optimizer.load_state_dict(resume["optimizer_state"])
        history, best_val, start_epoch = resume["history"], resume["best_val"], resume["epoch"] + 1
        train_generator.set_state(resume["train_generator_state"])
        torch.set_rng_state(resume["torch_rng_state"])
        if device.type == "cuda" and resume.get("cuda_rng_states") is not None:
            if len(resume["cuda_rng_states"]) == torch.cuda.device_count():
                torch.cuda.set_rng_state_all(resume["cuda_rng_states"])
            else:
                print("CUDA device count changed; continuing with newly seeded GPU random streams. "
                      "Training remains resumable, but its stochastic trajectory will change.", flush=True)
                torch.cuda.manual_seed_all(config["seed"] + start_epoch)
        # When resuming into a new directory, carry over the previously selected
        # best checkpoint if it is available next to the resumed checkpoint.
        if previous_best.resolve() != (output / "best.pt").resolve():
            import shutil
            shutil.copy2(previous_best, output / "best.pt")

    fixed_rng = torch.Generator().manual_seed(config["seed"] + 1)
    fixed_z = torch.randn(64, config["latent_dim"], generator=fixed_rng).to(device)
    preview_images = torch.stack([datasets["val"][i] for i in range(min(8, len(datasets["val"])))]).to(device)
    for epoch in range(start_epoch, args.epochs + 1):
        start_time = time.perf_counter()
        weight = config["kl_weight"]
        if config["kl_warmup_epochs"]:
            weight *= min(1.0, epoch / config["kl_warmup_epochs"])
        train_metrics = run_epoch(model, loaders["train"], device, weight, optimizer)
        # Reuse validation randomness every epoch and preserve the training RNG.
        # This makes changes in val loss easier to interpret on the tiny split.
        cuda_devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
        with torch.random.fork_rng(devices=cuda_devices):
            torch.manual_seed(config["seed"] + 1000)
            val_metrics = run_epoch(model, loaders["val"], device, config["kl_weight"])
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        duration = time.perf_counter() - start_time
        row = {"epoch": epoch, "seconds": duration, "train_kl_weight": weight,
               "val_kl_weight": config["kl_weight"],
               **{f"train_{key}": value for key, value in train_metrics.items()},
               **{f"val_{key}": value for key, value in val_metrics.items()}}
        history.append(row)
        with (output / "history.csv").open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(row))
            writer.writeheader()
            writer.writerows(history)
        is_best = val_metrics["total"] < best_val
        if is_best:
            best_val = val_metrics["total"]
        state = {"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                 "config": config, "epoch": epoch, "best_val": best_val, "history": history,
                 "manifest_digest": manifest_digest,
                 "torch_rng_state": torch.get_rng_state(),
                 "train_generator_state": train_generator.get_state(),
                 "cuda_rng_states": torch.cuda.get_rng_state_all() if device.type == "cuda" else None}
        # Write each checkpoint atomically so an interrupted write cannot replace
        # the last usable checkpoint with a partial file.
        for name in (["last.pt", "best.pt"] if is_best else ["last.pt"]):
            temporary = output / (name + ".tmp")
            torch.save(state, temporary)
            temporary.replace(output / name)
        with torch.inference_mode():
            model.eval()
            save_grid(model.decode(fixed_z), output / "previews" / f"samples_{epoch:03d}.png")
            mu, _ = model.encode(preview_images)
            reconstructed = model.decode(mu)
            pairs = torch.stack((preview_images, reconstructed), dim=1).flatten(0, 1)
            save_grid(pairs, output / "reconstructions.png", nrow=8)
        plot_history(history, output / "loss_curves.png")
        print(f"Epoch {epoch:03d}/{args.epochs}: train={train_metrics['total']:.6f}, "
              f"val={val_metrics['total']:.6f}, MSE={val_metrics['reconstruction']:.6f}, "
              f"KL={val_metrics['kl']:.3f}, {duration:.1f}s" + (" [best]" if is_best else ""), flush=True)
        print(f"Approximate remaining training/validation time at this rate: "
              f"{duration * (args.epochs - epoch) / 60:.1f} min (excludes plots/checkpoint writes).", flush=True)
    print(f"Finished. Selected model: {output / 'best.pt'}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hw1_vae/generate.py
"""Generate new images by decoding fresh standard-normal latent vectors."""

import argparse
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from .utils import load_model, resolve_device, save_grid, save_json


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, default=Path("runs/hw1_vae/generated"))
    parser.add_argument("--num-samples", type=int, default=64)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--seed", type=int, default=43)
    parser.add_argument("--device", choices=("auto", "cpu", "cuda", "mps"), default="auto")
    args = parser.parse_args()
    if args.num_samples < 1 or args.batch_size < 1:
        raise ValueError("num-samples and batch-size must be positive.")
    device = resolve_device(args.device)
    model, checkpoint = load_model(args.checkpoint, device)
    rng = torch.Generator().manual_seed(args.seed)
    if (args.output_dir / "generation.json").exists() or any(args.output_dir.glob("sample_*.png")):
        raise FileExistsError(f"{args.output_dir} already contains generated images. Choose a new "
                              "--output-dir so runs with different sample counts do not mix.")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    # The whole latent batch is drawn on CPU before chunking so changing the
    # decoder batch size preserves which latent vectors are generated.
    latents = torch.randn(args.num_samples, model.latent_dim, generator=rng)
    images = []
    with torch.inference_mode():
        for z in latents.split(args.batch_size):
            images.append(model.decode(z.to(device)).cpu())
    images = torch.cat(images)
    for index, tensor in enumerate(images):
        array = (tensor[0].clamp(0, 1).numpy() * 255).round().astype(np.uint8)
        Image.fromarray(array).save(args.output_dir / f"sample_{index:05d}.png")
    save_grid(images[:64], args.output_dir / "samples_grid.png")
    torch.save(latents, args.output_dir / "latents.pt")
    save_json(args.output_dir / "generation.json", {
        "checkpoint": str(args.checkpoint.resolve()), "checkpoint_epoch": checkpoint["epoch"],
        "num_samples": args.num_samples, "seed": args.seed,
        "method": "Draw z ~ N(0,I), display decoder mean; no input image or encoder is used.",
        "pixel_range": "[0,1] mapped linearly to 8-bit grayscale PNG", "image_size": model.image_size})
    print(f"Saved {len(images)} fresh samples and grid to {args.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hw1_vae/evaluate.py
"""Evaluate fresh prior samples with the homework's PyTorch-Ignite metrics.

This deliberately uses 1000 ImageNet class probabilities, as the default
extractor in the linked tutorial does. It is NOT the usual 2048-pool-feature
FID. Both metrics share one frozen pretrained Inception network.

Sources:
https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/
https://docs.pytorch.org/ignite/generated/ignite.metrics.FID.html
https://github.com/pytorch/ignite/blob/v0.5.3/ignite/metrics/gan/utils.py

Unlike the tutorial's GAN [-1,1] tensors, this VAE produces [0,1] images.
Preprocessing is explicit and identical for real and generated images:
repeat grayscale into RGB, bilinear resize to 299 with antialiasing, then
ImageNet normalization. These choices must accompany reported scores.
"""

from __future__ import annotations

import argparse
import importlib.metadata
import math
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset

from .data import ChestXrayDataset
from .utils import load_model, resolve_device, save_json


def preprocess_inception(images: torch.Tensor) -> torch.Tensor:
    """Map a batch N x 1 x H x W in [0,1] to normalized RGB299."""
    if images.ndim != 4 or images.shape[1] != 1 or images.shape[0] == 0:
        raise ValueError("Expected a nonempty N x 1 x H x W grayscale batch.")
    if not torch.isfinite(images).all():
        raise ValueError("Inception input contains nonfinite pixels.")
    if images.min() < 0 or images.max() > 1:
        raise ValueError("Expected pixels in [0,1] before Inception preprocessing.")
    rgb = images.float().repeat(1, 3, 1, 1)
    rgb = F.interpolate(
        rgb, size=(299, 299), mode="bilinear", align_corners=False, antialias=True
    )
    mean = rgb.new_tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
    std = rgb.new_tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)
    return (rgb - mean) / std


class InceptionProbabilities(nn.Module):
    """Frozen ImageNet Inception-v3; constructing it may download weights."""

    def __init__(self) -> None:
        super().__init__()
        from torchvision.models import Inception_V3_Weights, inception_v3

        # torchvision's transform_input adapts ImageNet-normalized input to
        # the normalization expected by the original Inception weights.
        self.network = inception_v3(
            weights=Inception_V3_Weights.IMAGENET1K_V1, transform_input=True
        )
        self.network.requires_grad_(False)
        self.network.eval()

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        logits = self.network(preprocess_inception(images))
        return torch.softmax(logits, dim=1)


def compute_feature_metrics(
    real_probabilities: torch.Tensor, generated_probabilities: torch.Tensor
) -> dict[str, float]:
    """Use Ignite on cached features; also supports tiny offline unit checks.

    Inputs are equal-size N x K probability matrices. Identity extractors
    avoid a second Inception forward and make the feature definition explicit.
    Ignite accumulates its moments in float64 and uses SciPy's matrix root.
    """
    from ignite.metrics import FID, InceptionScore

    if (
        real_probabilities.ndim != 2
        or generated_probabilities.shape != real_probabilities.shape
        or real_probabilities.shape[0] < 2
        or real_probabilities.shape[1] < 2
    ):
        raise ValueError("Expected matching N x K matrices with N >= 2 and K >= 2.")
    for probabilities in (real_probabilities, generated_probabilities):
        if not torch.isfinite(probabilities).all() or (probabilities < 0).any():
            raise ValueError("Class probabilities must be finite and nonnegative.")
        if not torch.allclose(
            probabilities.sum(dim=1),
            torch.ones_like(probabilities[:, 0]),
            atol=1e-5,
            rtol=1e-5,
        ):
            raise ValueError("Each probability row must sum to one.")

    # Accumulation on CPU avoids reserving GPU space for covariance matrices.
    real = real_probabilities.detach().cpu().double()
    generated = generated_probabilities.detach().cpu().double()
    dimensions = real.shape[1]
    fid_metric = FID(num_features=dimensions, feature_extractor=nn.Identity())
    generated_is = InceptionScore(num_features=dimensions, feature_extractor=nn.Identity())
    real_is = InceptionScore(num_features=dimensions, feature_extractor=nn.Identity())
    fid_metric.update((generated, real))
    generated_is.update(generated)
    real_is.update(real)
    raw_fid = float(fid_metric.compute())
    score = float(generated_is.compute())
    reference_score = float(real_is.compute())
    if not all(math.isfinite(value) for value in (raw_fid, score, reference_score)):
        raise FloatingPointError("Metric computation returned a nonfinite result.")
    if raw_fid < -1e-6:
        raise FloatingPointError(f"FID should be nonnegative; numerical result was {raw_fid}.")
    return {
        "fid": max(0.0, raw_fid),
        "fid_raw_before_roundoff_clamp": raw_fid,
        "inception_score": score,
        "real_inception_score_reference": reference_score,
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--data-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument(
        "--num-samples", type=int, default=0,
        help="Number of real/fresh generated images; 0 uses every official test image.",
    )
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--device", choices=("auto", "cpu", "cuda", "mps"), default="auto")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--num-workers", type=int, default=0)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.num_samples < 0 or args.batch_size < 1 or args.num_workers < 0:
        raise ValueError("num-samples/workers must be nonnegative; batch-size must be positive.")
    device = resolve_device(args.device)
    model, checkpoint = load_model(args.checkpoint, device)
    model.eval()
    test = ChestXrayDataset(args.data_dir, split="test", image_size=model.image_size)
    count = len(test) if args.num_samples == 0 else args.num_samples
    if not 2 <= count <= len(test):
        raise ValueError(f"num-samples must be between 2 and {len(test)}, or 0 for all test images.")

    # Separate streams keep test selection independent from prior draws.
    selection_rng = torch.Generator().manual_seed(args.seed)
    indices = torch.randperm(len(test), generator=selection_rng)[:count].tolist()
    prior_rng = torch.Generator(device="cpu").manual_seed(args.seed + 1)
    # Draw once before chunking, so decoder batch size cannot change which
    # latent vectors belong to this seeded evaluation.
    latents = torch.randn(count, model.latent_dim, generator=prior_rng)
    loader = DataLoader(
        Subset(test, indices), batch_size=args.batch_size, shuffle=False,
        num_workers=args.num_workers, pin_memory=device.type == "cuda", drop_last=False,
    )
    print("Loading frozen ImageNet Inception-v3 (first use downloads approximately 104 MB).", flush=True)
    extractor = InceptionProbabilities().to(device).eval()
    real_features: list[torch.Tensor] = []
    generated_features: list[torch.Tensor] = []
    seen = 0
    with torch.inference_mode():
        for batch in loader:
            images = batch[0] if isinstance(batch, (tuple, list)) else batch
            images = images.to(device, non_blocking=True)
            latent = latents[seen:seen + images.shape[0]].to(device)
            generated = model.decode(latent)
            real_features.append(extractor(images).cpu())
            generated_features.append(extractor(generated).cpu())
            seen += images.shape[0]
            print(f"Extracted features for {seen}/{count} real and generated images.", flush=True)
    # The network is no longer needed while SciPy computes the covariance root.
    del extractor
    values = compute_feature_metrics(torch.cat(real_features), torch.cat(generated_features))
    warnings = [
        "ImageNet features and labels differ from the chest X-ray domain; these scores do not establish medical validity.",
        "FID depends on sample count, preprocessing, weights and feature layer; compare only identical protocols.",
        "This is 1000-probability-feature FID following the Ignite tutorial, not standard 2048-pool-feature FID.",
    ]
    if count <= 1000:
        warnings.append(
            "There are fewer samples than feature dimensions, so empirical covariances are rank deficient and the finite-sample FID estimate is sensitive to sampling."
        )
    result = {
        **values,
        "checkpoint": str(args.checkpoint.resolve()),
        "checkpoint_epoch": checkpoint.get("epoch"),
        "data_dir": str(args.data_dir.resolve()),
        "reference_split": "test",
        "num_real_images": count,
        "num_generated_images": count,
        "batch_size": args.batch_size,
        "device": str(device),
        "model_image_size": model.image_size,
        "latent_dim": model.latent_dim,
        "reference_indices": indices,
        "seed": args.seed,
        "prior_seed": args.seed + 1,
        "generation": "z drawn independently from N(0,I); decoder means; no encoder/reconstruction input",
        "fid_feature_type": "1000 ImageNet class probabilities",
        "inception_weights": "torchvision Inception_V3_Weights.IMAGENET1K_V1",
        "preprocessing": {
            "source_pixels": "grayscale [0,1] at model resolution, for both real and generated",
            "rgb": "repeat grayscale channel three times",
            "resize": "torch interpolate to 299x299, bilinear, align_corners=False, antialias=True",
            "normalization_mean": [0.485, 0.456, 0.406],
            "normalization_std": [0.229, 0.224, 0.225],
            "torchvision_transform_input": True,
        },
        "directions": {"fid": "lower is better", "inception_score": "higher is conventionally better"},
        "inception_score_protocol": "one aggregate score over all generated samples, not mean/std of split scores",
        "versions": {
            package: importlib.metadata.version(package)
            for package in ("torch", "torchvision", "pytorch-ignite", "numpy", "scipy")
        },
        "limitations": warnings,
        "source": "https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/",
    }
    args.output_dir.mkdir(parents=True, exist_ok=True)
    output = args.output_dir / "metrics.json"
    save_json(output, result)
    print(f"FID (1000 probability features): {values['fid']:.6f}")
    print(f"Inception Score: {values['inception_score']:.6f}")
    print(f"Saved protocol and scores to {output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hw1_vae/smoke_test.py
"""Small CPU checks for the VAE pipeline, with no datasets or weights downloaded.

Run from the repository root with ``python -m hw1_vae.smoke_test``.
Temporary fixture images are artificial; passing these checks is not evidence
that a model has learned useful chest X-ray structure.
"""

from __future__ import annotations

import tempfile
import warnings
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from .data import CLASSES, SPLITS, ChestXrayDataset, resolve_data_root
from .evaluate import compute_feature_metrics, preprocess_inception
from .model import ConvVAE, vae_loss
from .utils import load_model


def check_data(temp_dir: Path) -> None:
    root = temp_dir / "extracted" / "chest_xray"
    pixels = np.arange(20 * 30, dtype=np.uint8).reshape(20, 30)
    for split in SPLITS:
        for class_name in CLASSES:
            class_dir = root / split / class_name
            class_dir.mkdir(parents=True)
            Image.fromarray(pixels).save(class_dir / "example.jpeg")
            # Invalid metadata files must never become training examples.
            (class_dir / "._example.jpeg").write_text("AppleDouble metadata")
            hidden_dir = class_dir / "__MACOSX"
            hidden_dir.mkdir()
            (hidden_dir / "ignored.jpeg").write_text("not an image")
        nested = root / split / "NORMAL" / "patient_subfolder"
        nested.mkdir()
        Image.fromarray(np.repeat(pixels[:, :, None], 3, axis=2)).save(nested / "rgb.png")
        # An extraction can also contain a nested duplicate dataset. It must
        # not be traversed when collecting images from the selected root.
        for class_name in CLASSES:
            duplicate = root / "chest_xray" / split / class_name
            duplicate.mkdir(parents=True)
            Image.fromarray(pixels).save(duplicate / "duplicate.jpeg")
    assert resolve_data_root(root.parent) == root
    assert resolve_data_root(root / "chest_xray") == root / "chest_xray"
    for split in SPLITS:
        dataset = ChestXrayDataset(root.parent, split, image_size=32)
        assert len(dataset) == 3
        assert dataset.class_counts == {"NORMAL": 2, "PNEUMONIA": 1}
        for path in dataset.image_paths:
            assert path.is_relative_to(root / split)
        image = dataset[0]
        assert image.shape == (1, 32, 32)
        assert image.dtype == torch.float32
        assert torch.isfinite(image).all() and 0 <= image.min() <= image.max() <= 1
        torch.testing.assert_close(image, dataset[0], rtol=0, atol=0)
    try:
        ChestXrayDataset(temp_dir / "does_not_exist", "train")
    except FileNotFoundError as error:
        assert "train/" in str(error) and "--data-dir" in str(error)
    else:
        raise AssertionError("Missing datasets must fail with a useful error")
    try:
        ChestXrayDataset(root, "training")
    except ValueError:
        pass
    else:
        raise AssertionError("Unknown split names must fail instead of mixing data")
    (root / "test" / "PNEUMONIA" / "example.jpeg").unlink()
    try:
        ChestXrayDataset(root, "test")
    except FileNotFoundError as error:
        assert "No supported image files" in str(error) and "PNEUMONIA" in str(error)
    else:
        raise AssertionError("Both diagnostic directories must contain images")
    print("PASS data: official splits, nested paths, duplicate/metadata exclusion, grayscale scaling")


def check_model_and_loss(temp_dir: Path) -> None:
    torch.manual_seed(42)
    config = {"image_size": 32, "latent_dim": 8, "base_channels": 8}
    model = ConvVAE(**config)
    for batch_size in (1, 4):
        model.zero_grad(set_to_none=True)
        images = torch.rand(batch_size, 1, config["image_size"], config["image_size"])
        reconstructed, mu, logvar = model(images)
        assert reconstructed.shape == images.shape
        assert mu.shape == logvar.shape == (batch_size, config["latent_dim"])
        assert torch.isfinite(reconstructed).all()
        assert 0 <= reconstructed.min() <= reconstructed.max() <= 1
        losses = vae_loss(reconstructed, images, mu, logvar)
        assert all(value.ndim == 0 and torch.isfinite(value) for value in losses.values())
        losses["total"].backward()
        for name, parameter in model.named_parameters():
            assert parameter.grad is not None, f"Missing gradient: {name}"
            assert torch.isfinite(parameter.grad).all(), f"Nonfinite gradient: {name}"
        for head in (model.fc_mu, model.fc_logvar):
            assert head.weight.grad.abs().sum() > 0, "Both posterior heads need a gradient"

    zeros = torch.zeros(4, config["latent_dim"])
    image = torch.rand(4, 1, 32, 32)
    prior_losses = vae_loss(image, image, zeros, zeros)
    torch.testing.assert_close(prior_losses["kl"], torch.tensor(0.0))
    torch.testing.assert_close(prior_losses["reconstruction"], torch.tensor(0.0))
    shifted_losses = vae_loss(image, image, torch.ones_like(zeros), zeros)
    torch.testing.assert_close(shifted_losses["kl"], torch.tensor(config["latent_dim"] / 2))
    random_losses = vae_loss(image * 0.8, image, torch.randn_like(zeros), torch.randn_like(zeros))
    assert random_losses["kl"] >= 0
    torch.testing.assert_close(
        random_losses["total"], random_losses["reconstruction"] + 0.001 * random_losses["kl"]
    )

    # Averaging across the batch must preserve the objective when identical
    # examples are repeated; this catches accidental batch-summed KL.
    one_mu, one_logvar = torch.randn(1, 8), torch.randn(1, 8)
    one_target, one_prediction = torch.rand(1, 1, 32, 32), torch.rand(1, 1, 32, 32)
    single = vae_loss(one_prediction, one_target, one_mu, one_logvar)
    repeated = vae_loss(
        one_prediction.repeat(4, 1, 1, 1), one_target.repeat(4, 1, 1, 1),
        one_mu.repeat(4, 1), one_logvar.repeat(4, 1),
    )
    for key in single:
        torch.testing.assert_close(single[key], repeated[key])

    checkpoint_path = temp_dir / "untrained_smoke_checkpoint.pt"
    torch.save({"model_state": model.state_dict(), "config": config, "epoch": 0}, checkpoint_path)
    restored, state = load_model(checkpoint_path, torch.device("cpu"))
    assert state["epoch"] == 0 and not restored.training
    latent = torch.randn(3, config["latent_dim"])
    model.eval()
    with torch.inference_mode():
        torch.testing.assert_close(model.decode(latent), restored.decode(latent), rtol=0, atol=0)
        full_size = ConvVAE(image_size=64, latent_dim=8, base_channels=8)
        output, _, _ = full_size(torch.rand(1, 1, 64, 64))
        assert output.shape == (1, 1, 64, 64)
    print("PASS model: batch sizes 1/4, 32/64px output, finite gradients, KL formula, checkpoint reload")


def check_metrics() -> None:
    # These tests use synthetic probabilities and an identity extractor. They
    # check metric plumbing without constructing Inception or fetching weights.
    from scipy.linalg import LinAlgWarning

    features = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
    with warnings.catch_warnings():
        # This analytic two-class fixture necessarily has singular covariance.
        # Suppress that specific expected warning only for this offline check.
        warnings.filterwarnings("ignore", message="Matrix is singular.*", category=LinAlgWarning)
        metrics = compute_feature_metrics(features, features)
    assert abs(metrics["fid"]) < 1e-6
    assert abs(metrics["inception_score"] - 2.0) < 1e-6
    processed = preprocess_inception(torch.zeros(2, 1, 32, 32))
    assert processed.shape == (2, 3, 299, 299) and torch.isfinite(processed).all()
    expected = -torch.tensor([0.485, 0.456, 0.406]) / torch.tensor([0.229, 0.224, 0.225])
    torch.testing.assert_close(processed[0, :, 0, 0], expected)
    print("PASS metrics: identical-set FID=0, known Inception Score=2, shared image preprocessing")


def main() -> None:
    torch.set_num_threads(2)
    with tempfile.TemporaryDirectory(prefix="ece285-vae-smoke-") as directory:
        temp_dir = Path(directory)
        check_data(temp_dir)
        check_model_and_loss(temp_dir)
        check_metrics()
    print("All offline CPU checks passed. No dataset or pretrained weights were downloaded.")


if __name__ == "__main__":
    main()


In [ ]:
run_module("hw1_vae.smoke_test")


## 8. Start a timed two-epoch pilot

The defaults are 64×64 grayscale, channels `[32,64,128,256]`, latent size
128, batch size 64, Adam learning rate 0.001, and KL weight 0.001. These are
starting choices. Higher resolution/width costs memory; increasing KL weight
pushes posteriors toward the prior more strongly. No augmentation or class
rebalancing is applied, so the empirical training mixture is retained.

Set `RUN_TRAINING=True` in the configuration cell, then rerun that cell and
this cell after preparing the data. Two epochs measure actual throughput;
do not infer GPU runtime from this Codespace's CPU checks. The printed epoch
time covers training/validation; plots and checkpoint writes add overhead.

Watch MSE, KL, and the fixed-latent image grids together. KL near zero with
nearly identical samples may indicate posterior collapse. Improve settings
using validation information, and reserve the test split for final scores.
For out-of-memory failures, start a fresh run with a smaller batch size.


In [ ]:
def require_training_setup():
    if not datasets:
        raise RuntimeError("Set DATA_DIR or download and rerun the dataset cell first.")
    if DEVICE.type != "cuda" and not ALLOW_CPU_TRAINING:
        raise RuntimeError("No CUDA kernel is selected. Move to your GPU runtime or explicitly "
                           "set ALLOW_CPU_TRAINING=True for CPU training.")
    if DEVICE.type == "cuda" and not compute_report["cuda_ready"]:
        raise RuntimeError("CUDA compute test failed; inspect compute_report before training.")

if RUN_TRAINING:
    require_training_setup()
    run_module("hw1_vae.train", "--data-dir", DATA_DIR, "--output-dir", RUN_DIR,
               "--device", DEVICE, "--epochs", PILOT_EPOCHS, "--batch-size", BATCH_SIZE,
               "--image-size", IMAGE_SIZE, "--latent-dim", LATENT_DIM,
               "--base-channels", BASE_CHANNELS, "--learning-rate", LEARNING_RATE,
               "--kl-weight", KL_WEIGHT, "--kl-warmup-epochs", KL_WARMUP_EPOCHS,
               "--seed", SEED, "--num-workers", NUM_WORKERS)
else:
    print("Fresh training skipped. Enable RUN_TRAINING after choosing data and compute.")


## 9. Resume on the same or another machine

Set `RUN_TRAINING=False`, `RUN_RESUME=True`, then rerun configuration and this
cell. `TOTAL_EPOCHS=50` means 50 epochs **in total**, including the pilot.
Existing checkpoints are not overwritten by a fresh run; use a new
`RUN_NAME` for a different experiment.

To move compute resources, copy the **whole run directory**, including
`last.pt` and `best.pt`, then change `RUN_DIR_OVERRIDE` and `DATA_DIR` to their
new absolute locations. The data manifest uses portable relative paths.
The same filenames/sizes/splits must remain available. Dataset manifests
are not content hashes and do not detect every image-content change.

Resume restores the saved architecture, optimizer, batch size, learning rate,
seed, and KL schedule. Changing those notebook variables does not override
an existing checkpoint. Start a new run for hyperparameter changes.
A CPU↔GPU or hardware/library change can change the stochastic trajectory;
portability does not mean bitwise equality between machines. Keep recorded
environment information when comparing runs.


In [ ]:
if RUN_RESUME:
    require_training_setup()
    run_module("hw1_vae.train", "--data-dir", DATA_DIR, "--output-dir", RUN_DIR,
               "--device", DEVICE, "--epochs", TOTAL_EPOCHS,
               "--resume", RUN_DIR / "last.pt", "--num-workers", NUM_WORKERS)
else:
    print("Resume skipped. Enable RUN_RESUME after the pilot, or after copying a run here.")


## 10. Read curves and inspect reconstructions

`history.csv` records train/validation total, MSE, KL, KL weights, and time.
`loss_curves.png` shows three panels. The total objective depends on the KL
weight, so a warmup run's training total is not directly comparable across
weights without considering its components.

`reconstructions.png` pairs each original with a deterministic reconstruction
made from its posterior mean, `decode(mu)`. This differs from the sampled
reconstruction used in the training/validation objective. The per-epoch
`previews/samples_###.png` grids use the same prior latents each epoch.


In [ ]:
if (RUN_DIR / "history.csv").is_file():
    import csv
    with (RUN_DIR / "history.csv").open(newline="") as handle:
        history_rows = list(csv.DictReader(handle))
    print("Latest epoch:", history_rows[-1])
    print("Settings:", json.loads((RUN_DIR / "config.json").read_text()))
    for name in ("loss_curves.png", "reconstructions.png"):
        display(Markdown(f"**{name}**"))
        display(DisplayImage(filename=str(RUN_DIR / name)))
    prior_previews = sorted((RUN_DIR / "previews").glob("samples_*.png"))
    if prior_previews:
        display(Markdown("**Latest fixed-latent prior samples**"))
        display(DisplayImage(filename=str(prior_previews[-1])))
else:
    print("No real training results yet. Curves and images appear after your pilot.")


## 11. Generate genuinely new images

To generate, draw $z\sim\mathcal N(0,I)$ and call `decode(z)` directly.
The encoder and a real input image are not involved. These fresh prior
samples are the generated examples required for the homework; reconstructions
must be labeled separately. Set `RUN_GENERATION=True` when `best.pt` exists.

The command saves individual PNGs, a grid, latent vectors, seed, and checkpoint
metadata. It uses a unique output folder for each invocation to keep runs
with different sample counts separate. Intensities use a fixed linear
`[0,1]`→`[0,255]` mapping with no per-image contrast normalization.


In [ ]:
if RUN_GENERATION:
    from datetime import datetime, timezone
    generation_dir = RUN_DIR / "generated" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_module("hw1_vae.generate", "--checkpoint", RUN_DIR / "best.pt",
               "--output-dir", generation_dir, "--num-samples", NUM_GENERATED,
               "--batch-size", BATCH_SIZE, "--device", DEVICE, "--seed", SEED + 1)
    display(DisplayImage(filename=str(generation_dir / "samples_grid.png")))
else:
    print("Generation skipped. Enable RUN_GENERATION after a checkpoint exists.")


## 12. Calculate the requested FID and Inception Score

**FID** compares feature means/covariances between real and generated images:
$\|m_r-m_g\|^2+\operatorname{Tr}(C_r+C_g-2(C_rC_g)^{1/2})$.
Lower is conventionally better. **IS** is
$\exp(\mathbb E_x\operatorname{KL}(p(y|x)\|p(y)))$:
it rewards confident predictions for each image and variety across images.
Higher is conventionally better; IS does not directly compare against real data.

Following the homework's
[Ignite tutorial](https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/),
this implementation explicitly uses **1000 ImageNet class probabilities**
for FID, not the common 2048-dimensional pooled-feature variant. It corrects
the tutorial's closing direction typo and documents its representation.
Do not compare our values with a different feature definition.

Both real and generated images start at model resolution in `[0,1]`, repeat
grayscale to RGB, resize to 299×299 with antialiased bilinear interpolation,
and receive identical ImageNet normalization. The frozen torchvision
Inception model returns class probabilities; Ignite computes the metrics.
Each test reference is matched by a fresh prior sample. We do not score
reconstructions or select a checkpoint using test FID.

Set `RUN_EVALUATION=True` after choosing the final checkpoint. First use
downloads about 104 MB of pretrained weights. `EVAL_NUM_SAMPLES=0` uses
the whole test split; a positive value selects that many without replacement.
The code saves score values **and** the complete evaluation protocol.

The ImageNet classifier was trained on natural-image categories. Its scores
do not demonstrate medical validity, and a small test set produces a
sample-sensitive, rank-deficient covariance estimate. Report sample count,
seed, feature definition, preprocessing, and limitations beside the scores.


In [ ]:
if RUN_EVALUATION:
    if not datasets:
        raise RuntimeError("Prepare the real dataset before final evaluation.")
    evaluation_dir = RUN_DIR / "evaluation"
    run_module("hw1_vae.evaluate", "--checkpoint", RUN_DIR / "best.pt",
               "--data-dir", DATA_DIR, "--output-dir", evaluation_dir,
               "--num-samples", EVAL_NUM_SAMPLES, "--batch-size", min(BATCH_SIZE, 32),
               "--device", DEVICE, "--seed", SEED, "--num-workers", NUM_WORKERS)
    display(JSON(json.loads((evaluation_dir / "metrics.json").read_text())))
else:
    print("Final metrics skipped. No score is claimed before training/evaluation.")


## 13. Save your work and prepare the report

Keep the notebook, generated source, checkpoint pair, configuration,
environment information, loss curves, fresh samples, and metrics protocol.
The export cell packages only the selected run and implementation; it omits
the dataset and virtual environment. Download/save the ZIP before a cloud
runtime disconnects, or use persistent storage for `RUN_DIR` from the start.

A notebook edit changes its embedded generated source. The repository CLI
is a separate copy; export the source actually used for the submitted run.
The report outline below is a plan with placeholders, not measured results.


In [ ]:
%%writefile REPORT_OUTLINE.md
# Homework 1 report plan

Use this outline to prepare the **two-page NeurIPS-format report** requested in the Homework 1 handout. Fill bracketed fields from the completed run. No training results have been supplied in this outline.

Use the course's template if one is provided. Otherwise, the [official NeurIPS 2026 page](https://neurips.cc/Conferences/2026/CallForPapers) links the [LaTeX author kit](https://media.neurips.cc/Conferences/NeurIPS2026/Formatting_Instructions_For_NeurIPS_2026.zip). Apply the homework's two-page limit; conference submission limits do not replace the assignment's instructions.

## Suggested page 1: task, method, and experiment

**Title:** A Convolutional Variational Autoencoder for Chest X-Ray Generation

**Abstract, about 50 words.** State the task, the unconditional convolutional VAE, the dataset, and the main measured outcome. Complete the outcome after training.

**Data and method, about 200 words.** Identify the [assigned dataset](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia), the actual train/validation/test counts, and the use of both diagnostic categories without label conditioning. Describe grayscale conversion, `64×64` resizing, and `[0,1]` scaling. Explain the Gaussian encoder, standard-normal prior, reparameterization, and decoder. Give the channel progression and latent dimension.

Write the implemented objective explicitly:

```text
R = squared reconstruction error averaged over images and pixels
K = Gaussian KL summed over latent coordinates, averaged over images
L = R + lambda * K
```

State the actual `lambda` and any warmup schedule. Identify this as the weighted MSE-plus-KL objective used for training. Mention the fixed-variance Gaussian motivation for MSE without claiming that the recorded scalar is a fully specified, unit-weight negative ELBO.

**Settings table.** Copy the actual run settings rather than assuming the defaults were used.

| Setting | Actual value |
| --- | --- |
| Resolution / channels | `[ ]` |
| Encoder channels / latent dimension | `[ ]` |
| Optimizer / learning rate | `[ ]` |
| Batch size / epochs completed | `[ ]` |
| KL weight / warmup | `[ ]` |
| Seed / training device | `[ ]` |
| Training duration / selected epoch | `[ ]` |

**Training figure.** Include training and validation curves. Show reconstruction MSE and KL as well as total loss, either as compact subplots or clearly labeled traces. If warmup changes the weight, identify the schedule so changes in total training loss are interpretable. Checkpoint selection uses validation data; report how the checkpoint was chosen.

## Suggested page 2: results and interpretation

**Generated-image figure.** Include a readable grid of fresh samples from `z ~ N(0,I)`. Label the model checkpoint and seed. A small reconstruction grid can be useful if space permits; label originals and reconstructions clearly.

**Quantitative results.** Fill in actual values from the evaluation JSON:

| Metric | Value | Evaluation set |
| --- | --- | --- |
| FID, tutorial-compatible 1000-probability representation | `[ ]` | `[N]` held-out real and `[N]` generated images |
| Inception Score | `[ ]` | Same `[N]` generated images |

Describe the Inception weights, resizing, grayscale-to-RGB conversion, input normalization, sample count, and seed as recorded by the evaluator. Cite the [assigned Ignite tutorial](https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/). State that this probability-feature FID variant differs from standard 2048-dimensional pooled-feature FID; only compare values computed using the same protocol.

**Discussion, about 150 words.** Explain what the curves and images actually show. Useful questions include: Is validation improving with training? Is KL close to zero? Are samples diverse? Are lung fields visible? Are reconstructions sharper than prior samples? Describe only observations supported by the saved outputs.

Discuss limitations relevant to the experiment: low image resolution, the small official validation split, sample-sensitive metric estimates, and the mismatch between an ImageNet classifier and chest X-ray semantics. A higher IS or lower FID alone does not establish anatomical or clinical validity.

**Closing sentence.** State the result achieved and one specific improvement suggested by the evidence.

Before submission, replace every placeholder, make figure labels readable at the final size, confirm the PDF has two pages, and include the runnable code. Follow any course-specific citation or AI-assistance disclosure instructions supplied separately.


# Homework 1 report plan

Use this outline to prepare the **two-page NeurIPS-format report** requested in the Homework 1 handout. Fill bracketed fields from the completed run. No training results have been supplied in this outline.

Use the course's template if one is provided. Otherwise, the [official NeurIPS 2026 page](https://neurips.cc/Conferences/2026/CallForPapers) links the [LaTeX author kit](https://media.neurips.cc/Conferences/NeurIPS2026/Formatting_Instructions_For_NeurIPS_2026.zip). Apply the homework's two-page limit; conference submission limits do not replace the assignment's instructions.

## Suggested page 1: task, method, and experiment

**Title:** A Convolutional Variational Autoencoder for Chest X-Ray Generation

**Abstract, about 50 words.** State the task, the unconditional convolutional VAE, the dataset, and the main measured outcome. Complete the outcome after training.

**Data and method, about 200 words.** Identify the [assigned dataset](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia), the actual train/validation/test counts, and the use of both diagnostic categories without label conditioning. Describe grayscale conversion, `64×64` resizing, and `[0,1]` scaling. Explain the Gaussian encoder, standard-normal prior, reparameterization, and decoder. Give the channel progression and latent dimension.

Write the implemented objective explicitly:

```text
R = squared reconstruction error averaged over images and pixels
K = Gaussian KL summed over latent coordinates, averaged over images
L = R + lambda * K
```

State the actual `lambda` and any warmup schedule. Identify this as the weighted MSE-plus-KL objective used for training. Mention the fixed-variance Gaussian motivation for MSE without claiming that the recorded scalar is a fully specified, unit-weight negative ELBO.

**Settings table.** Copy the actual run settings rather than assuming the defaults were used.

| Setting | Actual value |
| --- | --- |
| Resolution / channels | `[ ]` |
| Encoder channels / latent dimension | `[ ]` |
| Optimizer / learning rate | `[ ]` |
| Batch size / epochs completed | `[ ]` |
| KL weight / warmup | `[ ]` |
| Seed / training device | `[ ]` |
| Training duration / selected epoch | `[ ]` |

**Training figure.** Include training and validation curves. Show reconstruction MSE and KL as well as total loss, either as compact subplots or clearly labeled traces. If warmup changes the weight, identify the schedule so changes in total training loss are interpretable. Checkpoint selection uses validation data; report how the checkpoint was chosen.

## Suggested page 2: results and interpretation

**Generated-image figure.** Include a readable grid of fresh samples from `z ~ N(0,I)`. Label the model checkpoint and seed. A small reconstruction grid can be useful if space permits; label originals and reconstructions clearly.

**Quantitative results.** Fill in actual values from the evaluation JSON:

| Metric | Value | Evaluation set |
| --- | --- | --- |
| FID, tutorial-compatible 1000-probability representation | `[ ]` | `[N]` held-out real and `[N]` generated images |
| Inception Score | `[ ]` | Same `[N]` generated images |

Describe the Inception weights, resizing, grayscale-to-RGB conversion, input normalization, sample count, and seed as recorded by the evaluator. Cite the [assigned Ignite tutorial](https://pytorch-ignite.ai/blog/gan-evaluation-with-fid-and-is/). State that this probability-feature FID variant differs from standard 2048-dimensional pooled-feature FID; only compare values computed using the same protocol.

**Discussion, about 150 words.** Explain what the curves and images actually show. Useful questions include: Is validation improving with training? Is KL close to zero? Are samples diverse? Are lung fields visible? Are reconstructions sharper than prior samples? Describe only observations supported by the saved outputs.

Discuss limitations relevant to the experiment: low image resolution, the small official validation split, sample-sensitive metric estimates, and the mismatch between an ImageNet classifier and chest X-ray semantics. A higher IS or lower FID alone does not establish anatomical or clinical validity.

**Closing sentence.** State the result achieved and one specific improvement suggested by the evidence.

Before submission, replace every placeholder, make figure labels readable at the final size, confirm the PDF has two pages, and include the runnable code. Follow any course-specific citation or AI-assistance disclosure instructions supplied separately.


In [ ]:
if EXPORT_RESULTS:
    if not (RUN_DIR / "last.pt").is_file():
        raise RuntimeError("No training checkpoint exists to export yet.")
    import zipfile
    archive = WORKSPACE / f"{RUN_DIR.name}_results.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for source in sorted((WORKSPACE / "hw1_vae").glob("*.py")):
            bundle.write(source, f"source/hw1_vae/{source.name}")
        bundle.writestr("source/requirements.txt", "\n".join(DEPENDENCY_SPECS) + "\n")
        bundle.write(WORKSPACE / "REPORT_OUTLINE.md", "REPORT_OUTLINE.md")
        bundle.write(WORKSPACE / "compute_report.json", "compute_report.json")
        for artifact in sorted(RUN_DIR.rglob("*")):
            if (artifact.is_file() and artifact.suffix != ".tmp"
                    and artifact.resolve() != archive.resolve()):
                bundle.write(artifact, "run/" + artifact.relative_to(RUN_DIR).as_posix())
    print("Saved:", archive, "bytes:", archive.stat().st_size)
    print("Also save/download this notebook using your notebook application's File menu.")
    try:
        import google.colab
    except ImportError:
        print("Use your Jupyter file browser or operating system's file manager to "
              "save/download the ZIP at the printed absolute path.")
    else:
        from google.colab import files
        files.download(str(archive))
else:
    print("Export skipped. Enable EXPORT_RESULTS after training, before leaving the runtime.")


## Check your understanding

1. Why does the encoder output a mean and a log-variance for every image?
2. Where does randomness enter, and how does the gradient reach both heads?
3. What does zero KL mean, and why can minimizing KL alone discard information?
4. Why can reconstructions look better than fresh prior samples?
5. How would changing pixel-mean MSE to a sum affect the KL balance?
6. Which data split selects the checkpoint, and which one supplies final FID references?
7. What can ImageNet-based FID and IS tell you about chest X-ray anatomy?

For lecture review, revisit roughly 54–59 minutes for reparameterization
and 63–69 minutes for implementation. Record observations from your real
run before drawing conclusions about image quality.
